In [573]:
####################################
#ENVIRONMENT SETUP

In [574]:
#LIBRARIES

#system
import os
os.environ["HDF5_USE_FILE_LOCKING"] = "FALSE"
import sys

#math and array operations
import numpy as np
import pandas as pd
import math

#plotting
import matplotlib
# matplotlib.use("Agg") #UNCOMMENT IF PLOTTING WITHIN JUPYTER DOCUMENT
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from matplotlib.colors import TwoSlopeNorm

import cartopy.crs as ccrs
import cartopy.feature as cfeature

#data classes
import xarray as xr
import h5py
import pickle 

#loading bar
from tqdm import tqdm

#dates
from datetime import datetime

In [575]:
#Importing DirectoryManager Class
sys.path.append(os.path.join("/glade/u/home/aroseman/Projects/Regional-MPAS-Project/Code/CodeFiles/","DataAnalysis"))
from CLASSES_Directories import DirectoryManager_Class

In [576]:
DirectoryManager = DirectoryManager_Class()

codeType = os.path.join("DataAnalysis", "MPAS_Model_Data", "InitialFigures")
dataType = "Compare2mTemperature"

outputDirectory = DirectoryManager.GetOutputDirectory(codeType, dataType)
outputPlottingDirectory = DirectoryManager.GetOutputPlottingDirectory(codeType, dataType)

 DirectoryManager_Class Summary
 Main Directory:           /glade/u/home/aroseman/Projects/Regional-MPAS-Project
 Main Scratch Directory:   /glade/derecho/scratch/aroseman/Projects/Regional-MPAS-Project
 Scratch Directory:        /glade/derecho/scratch/aroseman/Projects/Regional-MPAS-Project/MPAS_Atmosphere_8.3.0
 Data Directory:           /glade/work/aroseman/Projects/Regional-MPAS-Project/Code/DATA
 Main Output Directory:    /glade/u/home/aroseman/Projects/Regional-MPAS-Project/Code/OUTPUT
 Main Output Plotting Directory:    /glade/u/home/aroseman/Projects/Regional-MPAS-Project/Code/PLOTTING
 Main Code Directory:      /glade/u/home/aroseman/Projects/Regional-MPAS-Project/Code/CodeFiles
 Current Code Directory:   /glade/u/home/aroseman/Projects/Regional-MPAS-Project/Code/CodeFiles/DataAnalysis/MPAS_Model_Data/InitialFigures



In [577]:
#Importing ModelData Class
sys.path.append(os.path.join(DirectoryManager.mainCodeDirectory,"DataAnalysis","MPAS_Model_Data"))
from CLASSES_ModelData import StructuredModelData_Class, DataOperator_Class

In [578]:
#Setup

# Region = "TRACER"; Case = "WET"; spinup_hours = "0"
# Region = "TRACER"; Case = "DIURNAL"; spinup_hours = "-5"

Region = "Hawaii"; Case = "TRADES"; spinup_hours = "12"

In [579]:
#Load Model Directory Class
RunType = (Region,Case,"NSSL",spinup_hours)
ModelData_NSSL = StructuredModelData_Class(DirectoryManager.mainDirectory, DirectoryManager.scratchDirectory, RunType)

RunType = (Region,Case,"TEMPO",spinup_hours)
ModelData_TEMPO = StructuredModelData_Class(DirectoryManager.mainDirectory, DirectoryManager.scratchDirectory, RunType)

Found 240/289 files matching expected times.
Opened history file: /glade/derecho/scratch/aroseman/Projects/Regional-MPAS-Project/MPAS_Atmosphere_8.3.0/Hawaii/TRADES/MPAS-Model_NSSL/model_run_spinup12hrs/history_cartesian/history.2022-08-07_12.00.00.latlon.nc
Opened diag file: /glade/derecho/scratch/aroseman/Projects/Regional-MPAS-Project/MPAS_Atmosphere_8.3.0/Hawaii/TRADES/MPAS-Model_NSSL/model_run_spinup12hrs/diag_cartesian/diag.2022-08-07_12.00.00.latlon.nc

=== MPAS Structured (lat-lon) Model Data Summary ===
 Region:         Hawaii
 Case:           TRADES
 Microphysics:   NSSL
 Resolution:     20-1km
 Time Step:      15mins
 Time Range:     2022-08-07 to 2022-08-10
 Coordinates:    ['latitude', 'longitude', 'nVertLevels', 'nVertLevelsP1']
 # History Files:240
 # Diag Files:   240
 # Time Steps:   240
 Data Directory: /glade/derecho/scratch/aroseman/Projects/Regional-MPAS-Project/MPAS_Atmosphere_8.3.0/Hawaii/TRADES/MPAS-Model_NSSL/model_run_spinup12hrs
 Static File:    Hawaii_region

In [580]:
#Importing ModelData Class
sys.path.append(os.path.join(DirectoryManager.mainCodeDirectory,"DataAnalysis"))
from CLASSES_DataSaving import DataSaving_Class

In [581]:
####################################
#OBSERVATION DATA LOADING

In [582]:
# TRACER DATA

# gndirt
# Description: Infrared Thermometer: Ground surface temperature
# Site: Houston, TX; Tracking Aerosol Convection interactions ExpeRiment (HOU)
# Location: Houston, TX; AMF1 (main site for TRACER) 
# Facility Code: M1
# Category: Radiometric
# Data Type: Routine Data 
# Source Instrument/Data: Infrared Thermometer 
# Start Date: 2021-08-04 
# End Date: 2022-10-01 

#CITATION:
# Atmospheric Radiation Measurement (ARM) user facility. 2021. Surface Meteorological Instrumentation (MET), 2022-06-08 to 2022-07-03, 
# ARM Mobile Facility (HOU) Houston, TX; AMF1 (main site for TRACER)  (M1).  

# Compiled by J. Kyrouac, Y. Shi and M. Tuftedal. ARM Data Center. Data set accessed 2025-11-04 at 
# https://urldefense.com/v3/__http://dx.doi.org/10.5439/1786358__;!!PvDODwlR4mBZyAb0!VeQa1CTGhZpEaT9OwHimXiqYCUp4161SPopymuMvEyM4RBhscXCdfhMBRa2JZcyEJusSMJHTm3OBm6qmrJdCJQ$


#DESCRIPTION
# https://armgov.svcs.arm.gov/capabilities/instruments/met
# The ARM Surface Meteorology Systems (MET) use mainly conventional in situ sensors to obtain 1-minute statistics of surface wind speed, wind direction, air temperature, relative humidity, barometric pressure, and rain-rate.
# Sensors may be added to or removed from the base set depending upon the deployment location, climate regime, or programmatic needs. Sensor types may also change depending upon the climate regime of the deployment.

#HANDBOOK
# https://www.arm.gov/publications/tech_reports/handbooks/met_handbook.pdf
# mentions that temperature probe is at standard height of 2 meters

def GetSurfaceData_TRACER():
    #getting dataPath
    def GetDataFolder(dataClassification,region,dataFolderName):
        dataPath = os.path.join(DirectoryManager.dataDirectory,dataClassification,region,dataFolderName)
        return dataPath
        
    dataClassification = "Observation_Data"
    region = ModelData_NSSL.region
    dataFolderName = "TRACER_MET"
    dataPath = GetDataFolder(dataClassification,region,dataFolderName)
    
    #reading data
    fileList, filePathList = DirectoryManager.ListFiles(dataPath)
    # xr.open_dataset(filePathList[5])["sfc_ir_temp"].plot()
    
    def GetTargetDates(ModelData):
        target_dates = [target_date.replace("-","") for target_date in ModelData.simulationDates][0:-1]
        return target_dates
        
    target_dates = GetTargetDates(ModelData_NSSL)
    ids = [i for i, f in enumerate(fileList) if any(date in f for date in target_dates)]
    
    surface_data = []; surface_time = []
    for count,i in enumerate(ids):
        currentFilePath = filePathList[i]
        
        # Open dataset and extract variable
        ds = xr.open_dataset(currentFilePath)
        var = ds["temp_mean"] #surface atmospheric measurement of mean temperature at 2 meters
        if count == 0:
            surfaceDataLat = ds['lat'].item()
            surfaceDataLon = ds['lon'].item()
        
        # Append data and time
        surface_data.append(var.data)
        surface_time.append(var["time"].data)
    
    # --- Combine all data and time ---
    surface_time = np.concatenate(surface_time)
    surface_data = np.concatenate(surface_data)+273.15

    stationLocation = "TRACER AMF1 Facility"
    stationDataName = "(MET Data)"
    return surface_time,surface_data, surfaceDataLat,surfaceDataLon, stationLocation,stationDataName

In [583]:
#MesoWest Weather Station Data

# MesoWest is a cooperative project to observe and archive mesoscale weather observations across the United States. 
# Their observations include, but are not limited to, temperature, humidity, wind speed, wind direction, and precipitation. 
# Their data is also known to be central for climate records, such as for monitoring microclimates.

# Data is collected from a variety of organizations. Some stations participate 
# in voluntary weather observing networks such as the Citizen Weather Observer Program (CWOP). 
# Others are part of formal mesonets[1] that are managed by private firms, federal/state/local agencies, and/or universities. 
# This data is utilized for a multitude of uses. Over 20,000 weather stations report to the MesoWest database.[2]

# MesoWest began as the Utah Mesonet, but was renamed as its scope expanded beyond the state. 
# Parties involved in this project include researchers at the University of Utah, forecasters at the Salt Lake City National Weather Service Forecast Office (NWSFO), 
# the National Weather Service Western Region Headquarters,[3] universities, and commercial firms. 
# Support for this project is being provided by the National Weather Service (NWS).

from scipy.ndimage import gaussian_filter1d
def GetSurfaceData_Hawaii(stationNumber):

    #getting dataPath
    def GetDataFolder(dataClassification,region,dataFolderName,ModelData):
        dataPath = os.path.join(DirectoryManager.dataDirectory,dataClassification,region,dataFolderName)
        return dataPath
        
    dataClassification = "Observation_Data"
    region = ModelData_NSSL.region
    dataFolderName = "NWS_MesoWest_SurfaceData"
    dataPath = GetDataFolder(dataClassification,region,dataFolderName,ModelData_NSSL)
    dataPath = os.path.join(dataPath, ModelData_NSSL.case)
    
    #reading data
    
    fileList, filePathList = DirectoryManager.ListFiles(dataPath)

    def ConvertToUTC(time_strings):
        
        # Remove the literal " HST" from each timestamp
        clean_strings = [t.replace(" HST", "") for t in time_strings]
        
        # Parse to datetime (naive)
        times_dt = pd.to_datetime(clean_strings, format="%m-%d-%Y %H:%M")
        
        # Localize to HST (Pacific/Honolulu)
        times_lt = times_dt.tz_localize("Pacific/Honolulu")
        
        # Convert to UTC
        times_utc = times_lt.tz_convert("UTC")
        return times_utc
    
    def GetData(filePathList_Station):
        times, temperatures = [],[]
        for filePaths in filePathList_Station:
            df = pd.read_excel(filePaths)
            ts_col = df.columns[0]
            
            # Remove footer rows that contain known footer text
            footer_phrases = ["Contact", "Data provided"]
            
            mask = df[ts_col].astype(str).str.contains('|'.join(footer_phrases), case=False, na=False)
            
            df = df[~mask]   # keep rows that do NOT contain footer text
            # df["TMP ° C"] = pd.to_numeric(df["TMP ° C"], errors="coerce")
            
            time_list = df[ts_col].tolist()[::-1]
            temp_list = df["TMP ° C"].tolist()[::-1]
        
            times.extend(time_list)
            temperatures.extend(temp_list)
        temperatures = [round(temperature + 273.15, 2) for temperature in temperatures]

        times = ConvertToUTC(times)
        times =times.tz_localize(None).to_numpy(dtype="datetime64[ns]")
        return times, np.array(temperatures)

    def GetStationInfo(stationName):
        
        if stationName == "PHLI":
            #https://forecast.weather.gov/MapClick.php?lon=-159.35435393142717&lat=21.98123584309586
            (stationLat,stationLon) = 21.98,-159.34
            stationLocation = "Lihue, Kauai, Hawaii"
        elif stationName == "PHMK":
            #https://forecast.weather.gov/MapClick.php?lat=21.1529&lon=-157.0963
            (stationLat,stationLon) = 21.15,-157.1
            stationLocation = "Kaunakakai, Molokai, Hawaii"
        elif stationName == "PHTO":
            #https://forecast.weather.gov/MapClick.php?lat=19.7203&lon=-155.0485
            (stationLat,stationLon) = 19.72, -155.06 
            stationLocation = "Hilo, Big Island, Hawaii"
        
        return stationLat,stationLon, stationName,stationLocation

    filePathList_Station1 = [f for f in filePathList if os.path.basename(f).startswith("1_PHLI")]
    filePathList_Station2 = [f for f in filePathList if os.path.basename(f).startswith("2_PHMK")]
    filePathList_Station3 = [f for f in filePathList if os.path.basename(f).startswith("3_PHTO")]
    [surface_time1, surface_data1] = GetData(filePathList_Station1)
    [surface_time2, surface_data2] = GetData(filePathList_Station2)
    [surface_time3, surface_data3] = GetData(filePathList_Station3)

    stationDataName = "MesoWest Data"
    
    if stationNumber == 1:
        [surface_time, surface_data] = GetData(filePathList_Station1)
        stationLat,stationLon, stationName,stationLocation = GetStationInfo(stationName="PHLI")
    elif stationNumber == 2:
        [surface_time, surface_data] = GetData(filePathList_Station2)
        stationLat,stationLon, stationName,stationLocation = GetStationInfo(stationName="PHMK")
    elif stationNumber == 3:
        [surface_time, surface_data] = GetData(filePathList_Station3)
        stationLat,stationLon, stationName,stationLocation = GetStationInfo(stationName="PHTO")
    
    return surface_time,surface_data,stationLat,stationLon, stationLocation,stationDataName

In [584]:
if ModelData_NSSL.region == "TRACER":
    [surface_time1,surface_data1, surfaceDataLat1,surfaceDataLon1, stationLocation1,stationDataName1] = GetSurfaceData_TRACER()
elif ModelData_NSSL.region == "Hawaii":
    [surface_time1,surface_data1, surfaceDataLat1,surfaceDataLon1, stationLocation1,stationDataName1] = GetSurfaceData_Hawaii(stationNumber=1)
    surface_data1 = gaussian_filter1d(np.array(surface_data1), sigma=2) #only needed for Hawaii Data

    [surface_time2,surface_data2, surfaceDataLat2,surfaceDataLon2, stationLocation2,stationDataName2] = GetSurfaceData_Hawaii(stationNumber=2)
    surface_data2 = gaussian_filter1d(np.array(surface_data2), sigma=2) #only needed for Hawaii Data

    [surface_time3,surface_data3, surfaceDataLat3,surfaceDataLon3, stationLocation3,stationDataName3] = GetSurfaceData_Hawaii(stationNumber=3)
    surface_data3 = gaussian_filter1d(np.array(surface_data3), sigma=2) #only needed for Hawaii Data

In [585]:
####################################
#MODEL DATA LOADING

In [586]:
def findNearestTimes(reference_times, times):
    """
    For each time in surface_time, find the index of the closest time in model_times.
    """
    
    # Use broadcasting to find the absolute difference and take the argmin
    idx = np.abs(reference_times[:, None] - times[None, :]).argmin(axis=1)
    
    return idx.tolist()

time_strings = [t.replace(":", ".") for t in ModelData_NSSL.timeStrings]
model_times = [datetime.strptime(t, "%Y-%m-%d_%H.%M.%S") for t in time_strings]
model_times = np.array(model_times, dtype='datetime64[ns]')
closest_times = findNearestTimes(reference_times=surface_time, times=model_times)

In [587]:
def GetDataTimestep_cached(ModelData, t, varName, cache):
    """
    Retrieve model variable for a given timestep using an in-memory cache.
    """
    # If time already loaded, return from cache
    if t in cache:
        return cache[t]
    else:
        print(f"loading for time {t}")
    
    # Otherwise, load and store it
    data = ModelData.GetDataTimestep_diag(t=t, varName=varName)
    cache[t] = data
    return data


In [588]:
def GetModelSurfaceData(ModelData, closest_times, varName, lat, lon, extra=""):
    """
    Loads model surface data if cached, otherwise computes and saves it.
    """

    inputDirectory = os.path.join(outputDirectory,(
        f"modelSurfaceData_t2m_{ModelData.region}_"
        f"{ModelData.case}_{ModelData.mpType}_"
        f"spinup{ModelData.spinup_hours}hrs{extra}.pkl" #extra is if using more stations (i.e. Hawaii case)
    ))

    # --- 1. If pickle file exists, load it ---
    if os.path.exists(inputDirectory):
        print(f"Loading cached model surface data from {inputDirectory}")
        with open(inputDirectory, "rb") as f:
            data = pickle.load(f)

        return np.array(data["values"]), data["times"]

    # --- 2. Otherwise, compute and save ---
    print("Cache not found — computing model surface data...")
    time_cache = {}
    modelSurfaceData = []
    modelTimes = []

    for t in tqdm(closest_times):
        data_t = GetDataTimestep_cached(ModelData, t=t, varName=varName, cache=time_cache)
        selection = data_t.sel(latitude=lat, longitude=lon, method="nearest").data
        modelSurfaceData.append(selection)

        # Store the model time for this timestep
        modelTimes.append(ModelData.timeStrings[t])

    # Convert surface data to NumPy array
    modelSurfaceData = np.array(modelSurfaceData)

    # --- 3. Save BOTH data + times ---
    cache_to_save = {
        "values": modelSurfaceData,
        "times": modelTimes
    }

    with open(inputDirectory, "wb") as f:
        pickle.dump(cache_to_save, f)

    print(f"Saved computed model surface data to {inputDirectory}")

    return modelSurfaceData, modelTimes


In [589]:
if ModelData_NSSL.region == "TRACER":
    
    #loading for NSSL
    [modelSurfaceData_NSSL1,modelTimes] = GetModelSurfaceData(
        ModelData=ModelData_NSSL,
        closest_times=closest_times,
        varName="t2m",
        lat=surfaceDataLat1,
        lon=surfaceDataLon1)
    
    #loading for TEMPO
    [modelSurfaceData_TEMPO1,_] = GetModelSurfaceData(
        ModelData=ModelData_TEMPO,
        closest_times=closest_times,
        varName="t2m",
        lat=surfaceDataLat1,
        lon=surfaceDataLon1)

In [590]:
if ModelData_NSSL.region == "Hawaii":
    #loading for NSSL
    [modelSurfaceData_NSSL1,modelTimes] = GetModelSurfaceData(
        ModelData=ModelData_NSSL,
        closest_times=closest_times,
        varName="t2m",
        lat=surfaceDataLat1,
        lon=surfaceDataLon1,
        extra="_station1")

    [modelSurfaceData_NSSL2,modelTimes] = GetModelSurfaceData(
        ModelData=ModelData_NSSL,
        closest_times=closest_times,
        varName="t2m",
        lat=surfaceDataLat2,
        lon=surfaceDataLon2,
        extra="_station2")

    [modelSurfaceData_NSSL3,modelTimes] = GetModelSurfaceData(
        ModelData=ModelData_NSSL,
        closest_times=closest_times,
        varName="t2m",
        lat=surfaceDataLat3,
        lon=surfaceDataLon3,
        extra="_station3")
    
    #loading for TEMPO
    [modelSurfaceData_TEMPO1,_] = GetModelSurfaceData(
        ModelData=ModelData_TEMPO,
        closest_times=closest_times,
        varName="t2m",
        lat=surfaceDataLat1,
        lon=surfaceDataLon1,
        extra="_station1")
    
    [modelSurfaceData_TEMPO2,_] = GetModelSurfaceData(
        ModelData=ModelData_TEMPO,
        closest_times=closest_times,
        varName="t2m",
        lat=surfaceDataLat2,
        lon=surfaceDataLon2,
        extra="_station2")

    [modelSurfaceData_TEMPO3,_] = GetModelSurfaceData(
        ModelData=ModelData_TEMPO,
        closest_times=closest_times,
        varName="t2m",
        lat=surfaceDataLat3,
        lon=surfaceDataLon3,
        extra="_station3")

loading for time 29
Opened diag file: /glade/derecho/scratch/aroseman/Projects/Regional-MPAS-Project/MPAS_Atmosphere_8.3.0/Hawaii/TRADES/MPAS-Model_TEMPO/model_run_spinup12hrs/diag_cartesian/diag.2022-08-07_19.15.00.latlon.nc


 10%|█         | 128/1273 [00:12<02:33,  7.47it/s]

loading for time 30
Opened diag file: /glade/derecho/scratch/aroseman/Projects/Regional-MPAS-Project/MPAS_Atmosphere_8.3.0/Hawaii/TRADES/MPAS-Model_TEMPO/model_run_spinup12hrs/diag_cartesian/diag.2022-08-07_19.30.00.latlon.nc


 10%|█         | 131/1273 [00:12<02:29,  7.64it/s]

loading for time 31
Opened diag file: /glade/derecho/scratch/aroseman/Projects/Regional-MPAS-Project/MPAS_Atmosphere_8.3.0/Hawaii/TRADES/MPAS-Model_TEMPO/model_run_spinup12hrs/diag_cartesian/diag.2022-08-07_19.45.00.latlon.nc


...

Opened diag file: /glade/derecho/scratch/aroseman/Projects/Regional-MPAS-Project/MPAS_Atmosphere_8.3.0/Hawaii/TRADES/MPAS-Model_TEMPO/model_run_spinup12hrs/diag_cartesian/diag.2022-08-09_13.45.00.latlon.nc
loading for time 199
Opened diag file: /glade/derecho/scratch/aroseman/Projects/Regional-MPAS-Project/MPAS_Atmosphere_8.3.0/Hawaii/TRADES/MPAS-Model_TEMPO/model_run_spinup12hrs/diag_cartesian/diag.2022-08-09_14.00.00.latlon.nc
loading for time 200
Opened diag file: /glade/derecho/scratch/aroseman/Projects/Regional-MPAS-Project/MPAS_Atmosphere_8.3.0/Hawaii/TRADES/MPAS-Model_TEMPO/model_run_spinup12hrs/diag_cartesian/diag.2022-08-09_14.15.00.latlon.nc
loading for time 201
Opened diag file: /glade/derecho/scratch/aroseman/Projects/Regional-MPAS-Project/MPAS_Atmosphere_8.3.0/Hawaii/TRADES/MPAS-Model_TEMPO/model_run_spinup12hrs/diag_cartesian/diag.2022-08-09_14.30.00.latlon.nc
loading for time 202
Opened diag file: /glade/derecho/scratch/aroseman/Projects/Regional-MPAS-Project/MPAS_Atmosp

 63%|██████▎   | 803/1273 [00:02<00:01, 348.32it/s]

loading for time 220
Opened diag file: /glade/derecho/scratch/aroseman/Projects/Regional-MPAS-Project/MPAS_Atmosphere_8.3.0/Hawaii/TRADES/MPAS-Model_TEMPO/model_run_spinup12hrs/diag_cartesian/diag.2022-08-09_19.15.00.latlon.nc
loading for time 221
Opened diag file: /glade/derecho/scratch/aroseman/Projects/Regional-MPAS-Project/MPAS_Atmosphere_8.3.0/Hawaii/TRADES/MPAS-Model_TEMPO/model_run_spinup12hrs/diag_cartesian/diag.2022-08-09_19.30.00.latlon.nc
loading for time 222
Opened diag file: /glade/derecho/scratch/aroseman/Projects/Regional-MPAS-Project/MPAS_Atmosphere_8.3.0/Hawaii/TRADES/MPAS-Model_TEMPO/model_run_spinup12hrs/diag_cartesian/diag.2022-08-09_19.45.00.latlon.nc
loading for time 223
Opened diag file: /glade/derecho/scratch/aroseman/Projects/Regional-MPAS-Project/MPAS_Atmosphere_8.3.0/Hawaii/TRADES/MPAS-Model_TEMPO/model_run_spinup12hrs/diag_cartesian/diag.2022-08-09_20.00.00.latlon.nc
loading for time 224
Opened diag file: /glade/derecho/scratch/aroseman/Projects/Regional-MPA

100%|██████████| 1273/1273 [00:02<00:00, 498.46it/s]


Saved computed model surface data to /glade/u/home/aroseman/Projects/Regional-MPAS-Project/Code/OUTPUT/DataAnalysis/MPAS_Model_Data/InitialFigures/Compare2mTemperature/modelSurfaceData_t2m_Hawaii_TRADES_TEMPO_spinup12hrs_station3.pkl


In [591]:
#Getting Model Time
model_time = np.array([ts.replace("_", "T").replace(".", ":") for ts in modelTimes],dtype="datetime64[ns]")

In [592]:
####################################
#PLOTTING FUNCTIONS

In [593]:
def PlotSurfaceComparison(surface_time, surface_data,
                          model_time, model_data,
                          variable_name="Surface Temperature", units="K",
                          site_name="Houston (TRACER)", model_label="MPAS",
                          fig=None,ax=None,linestyle='solid'):

    import matplotlib.gridspec as gridspec

    # Create figure with GridSpec only when starting a new figure
    if (fig is None) or (ax is None):
        fig = plt.figure(figsize=(10, 6))
        gs = gridspec.GridSpec(2, 1, height_ratios=[0.12, 0.88])

        ax_top    = fig.add_subplot(gs[0])   # area for title + model legend
        ax        = fig.add_subplot(gs[1])   # main plot axis
    else:
        # If figure/ax passed, assume only single-axis usage
        # (will still work, but no top panel)
        ax_top = None

    # --- Plot observations ---
    ax.plot(surface_time, surface_data, color='black', linestyle=linestyle, linewidth=1.8,
            label=f"({site_name})")

    # --- Get Time Limits ---
    tmin, tmax = pd.to_datetime(model_time.min()), pd.to_datetime(model_time.max())

    # Normalize model inputs
    if not isinstance(model_data, (list, tuple)):
        model_data = [model_data]
    if not isinstance(model_time, (list, tuple)):
        model_time = [model_time] * len(model_data)
    if isinstance(model_label, str):
        model_label = [model_label] * len(model_data)

    # --- Plot models ---
    colors = plt.cm.tab10.colors
    for i, (m_time, m_data, label) in enumerate(zip(model_time, model_data, model_label)):
        ax.plot(m_time, m_data, color=colors[i], linewidth=1.6, linestyle=linestyle)

    # --- Formatting ---
    ax.set_xlabel("Time (UTC)", fontsize=12)
    ax.set_ylabel(f"{variable_name} [{units}]", fontsize=12)
    ax.set_xlim(tmin, tmax)

    ax.xaxis.set_major_formatter(mdates.DateFormatter("%m-%d\n%H:%M"))
    fig.autofmt_xdate()
    ax.grid(True, linestyle="--", alpha=0.4)

    # --- Axes legend (OBS only) ---
    ax.legend(frameon=False, fontsize=10, loc='lower left')

    # ---- TOP PANEL: SUPTITLE + MODEL LEGEND ----
    if ax_top is not None:
        ax_top.axis("off")  # remove axes lines

        # Title
        # ax_top.text(0.5, 0.75, f"{variable_name} Comparison",
        #              ha='center', va='center', fontsize=15, weight='bold')

        # Model Legend
        model_handles = [
            plt.Line2D([0], [0], color=colors[i], linestyle='-', linewidth=2)
            for i in range(len(model_label))
        ]

        ax_top.legend(
            handles=model_handles,
            labels=model_label,
            loc='lower center',
            ncol=len(model_label),
            fontsize=10,
            frameon=False
        )

    fig.tight_layout()

    return fig, ax


In [594]:
def SaveFigure(fig, dataType, dpi=150, extension="png"):
    """
    Saves a matplotlib Figure to a subdirectory named after the model configuration.
    """

    # Ensure dpi is a plain Python int
    dpi = int(np.atleast_1d(dpi)[0])  # Handles np.float64 or array inputs safely

    # --- Define output subdirectory ---
    outputSubDirectory = f"{ModelData_NSSL.region}_{ModelData_NSSL.case}_{ModelData_NSSL.mpType}vs{ModelData_TEMPO.mpType}_{ModelData_NSSL.spinup_hours}hrs"
    save_dir = os.path.join(outputPlottingDirectory, outputSubDirectory)
    os.makedirs(save_dir, exist_ok=True)

    # --- File path ---
    outputFile = os.path.join(
        save_dir,
        f"{dataType}.{extension}"
    )

    # --- Save and close ---
    fig.savefig(outputFile, dpi=dpi, bbox_inches="tight")
    plt.close(fig)
    print(f"Saved figure to: {outputFile}")

In [595]:
####################################
#PLOTTING

In [ ]:
fig,ax = PlotSurfaceComparison(
    surface_time=surface_time1,
    surface_data=surface_data1,
    model_time=model_time, 
    model_data=[modelSurfaceData_NSSL1,modelSurfaceData_TEMPO1],
    variable_name="2m Temperature",
    units="K",
    site_name=f"{stationLocation1}",
    model_label=[f"{ModelData_NSSL.region}_{ModelData_NSSL.case}_NSSL",f"{ModelData_NSSL.region}_{ModelData_NSSL.case}_TEMPO"]
)

if ModelData_NSSL.region == "Hawaii":
    fig,ax = PlotSurfaceComparison(
        surface_time=surface_time2,
        surface_data=surface_data2,
        model_time=model_time, 
        model_data=[modelSurfaceData_NSSL2,modelSurfaceData_TEMPO2],
        variable_name="2m Temperature",
        units="K",
        site_name=f"{stationLocation2}",
        model_label=[f"{ModelData_NSSL.region}_{ModelData_NSSL.case}_NSSL",f"{ModelData_NSSL.region}_{ModelData_NSSL.case}_TEMPO"],
        fig=fig,ax=ax,linestyle="dashed"
    )   

    fig,ax = PlotSurfaceComparison(
        surface_time=surface_time3,
        surface_data=surface_data3,
        model_time=model_time, 
        model_data=[modelSurfaceData_NSSL3,modelSurfaceData_TEMPO3],
        variable_name="2m Temperature",
        units="K",
        site_name=f"{stationLocation3}",
        model_label=[f"{ModelData_NSSL.region}_{ModelData_NSSL.case}_NSSL",f"{ModelData_NSSL.region}_{ModelData_NSSL.case}_TEMPO"],
        fig=fig,ax=ax,linestyle="dotted"
    )   


SaveFigure(fig, dataType)